# generator-project-and-reshape — ex2: project latent → 8x8 seed → two ConvTranspose2d upsamples to 32x32

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `generator-project-and-reshape`. Running the final beacon cell reports progress against the `GAN: Generator project + reshape` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: Generator project + reshape` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`generator-project-and-reshape`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "generator-project-and-reshape"
DD_SUBTOPIC = "GAN: Generator project + reshape"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Project + reshape to 8x8 seed, then 2 upsamples to 32x32

Ex1 projected latent `z` → 4x4 spatial seed. The deepening move is a
DIFFERENT seed size (8x8) followed by two `ConvTranspose2d` upsamples
to land at 32x32. The intermediate shapes are the load-bearing thing.

```
z: (B, latent_dim)
  -> Linear(latent_dim, C*8*8) -> view (B, C, 8, 8)        # spatial seed
  -> ConvTranspose2d(C, C//2, k=4, s=2, p=1) -> (B, C//2, 16, 16)
  -> ConvTranspose2d(C//2, out_C, k=4, s=2, p=1) -> (B, out_C, 32, 32)
```

**Why 8x8 not 4x4.** A larger seed gives more spatial info to the
early layers — fewer aggressive 2x upsamples needed. Pix2Pix and many
DCGAN variants pick the seed to match `final_H >> n_upsamples` — at
32x32 with 2 upsamples that's `32 >> 2 = 8`.

**Seed reshape pattern.** `Linear(latent_dim, C*H*W).view(B, C, H, W)`.
The Linear is doing all three jobs: dimensionality blow-up, learned
spatial layout, AND the only non-conv mixing in the generator.

### Exercise 2 — project latent → 8x8 seed → two ConvTranspose2d upsamples to 32x32

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply a `Linear(latent_dim, base_C * 8 * 8)` projection + `view` to produce an 8x8 spatial seed, then two `ConvTranspose2d(k=4, s=2, p=1)` upsamples to land at the requested 32x32 output, exposing each intermediate shape for verification.
> Keywords: generator, convtranspose2d, upsample, shapes
> ```

**KCs targeted:** `linear-project-then-view-to-spatial-seed`, `convtranspose-2x-upsample-k4-s2-p1`

Implement `ex2_Generator832`, an `nn.Module`. The deepening of ex1's 4x4 seed: now we use an **8x8** seed and TWO upsamples to reach **32x32**.

`__init__(latent_dim: int, base_C: int = 64, out_C: int = 3)` creates:
1. `self.project = nn.Linear(latent_dim, base_C * 8 * 8)`.
2. `self.up1 = nn.ConvTranspose2d(base_C, base_C // 2, kernel_size=4, stride=2, padding=1)` — doubles spatial.
3. `self.up2 = nn.ConvTranspose2d(base_C // 2, out_C, kernel_size=4, stride=2, padding=1)` — doubles again.
Store `self.base_C` for the view step.

`forward(z: Tensor) -> dict` where `z: (B, latent_dim)`:
1. `h = self.project(z)` — shape `(B, base_C * 64)`.
2. `seed = h.view(B, base_C, 8, 8)`.
3. `mid = self.up1(seed)` — shape `(B, base_C // 2, 16, 16)`.
4. `out = self.up2(mid)` — shape `(B, out_C, 32, 32)`.
5. Return `{'seed': seed, 'mid': mid, 'out': out}` (intermediate shapes are the testable thing).

No activations needed between the upsamples for this drill — we're verifying the shape pipeline, not the full DCGAN block.

In [ ]:
class ex2_Generator832(nn.Module):
    def __init__(self, latent_dim: int, base_C: int = 64, out_C: int = 3):
        super().__init__()
        self.base_C  = base_C
        self.project = nn.Linear(latent_dim, base_C * 8 * 8)
        self.up1     = nn.ConvTranspose2d(base_C, base_C // 2, kernel_size=4, stride=2, padding=1)
        self.up2     = nn.ConvTranspose2d(base_C // 2, out_C, kernel_size=4, stride=2, padding=1)

    def forward(self, z: Tensor) -> dict:
        B = z.shape[0]
        h    = self.project(z)
        seed = h.view(B, self.base_C, 8, 8)
        mid  = self.up1(seed)
        out  = self.up2(mid)
        return {'seed': seed, 'mid': mid, 'out': out}


<details><summary>Solution</summary>

```python
class ex2_Generator832(nn.Module):
    def __init__(self, latent_dim: int, base_C: int = 64, out_C: int = 3):
        super().__init__()
        self.base_C  = base_C
        self.project = nn.Linear(latent_dim, base_C * 8 * 8)
        self.up1     = nn.ConvTranspose2d(base_C, base_C // 2, kernel_size=4, stride=2, padding=1)
        self.up2     = nn.ConvTranspose2d(base_C // 2, out_C, kernel_size=4, stride=2, padding=1)

    def forward(self, z: Tensor) -> dict:
        B = z.shape[0]
        h    = self.project(z)
        seed = h.view(B, self.base_C, 8, 8)
        mid  = self.up1(seed)
        out  = self.up2(mid)
        return {'seed': seed, 'mid': mid, 'out': out}
```

**`k=4, s=2, p=1` exactly doubles spatial.** Output size formula for ConvTranspose2d is `H_out = (H_in - 1)*s - 2*p + k`. With `s=2, p=1, k=4`: `H_out = 2*H_in - 2 + 4 - 2 = 2*H_in`. Clean doubling — the canonical DCGAN upsample block.

**Why an 8x8 seed instead of 4x4.** With `final_H = 32` and the canonical 2x upsamples, you need `log2(32/seed_H)` upsamples. `seed=8` → 2 upsamples; `seed=4` → 3 upsamples. Larger seed = shallower decoder = easier to train at the cost of more Linear parameters in the projection.

**`base_C` lives on `self` because forward needs it.** You could infer it from `self.project.out_features // 64` instead — same result, slightly less clear. Storing it explicit is the standard practice in ARENA / DCGAN code.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()